##Install Requirement

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/alexandrachirita98/MedViT-Quantum/

In [ ]:
%cd /kaggle/working/MedViT-Quantum

In [ ]:
%pwd

In [ ]:
pip install -r requirements.txt

In [ ]:
print("PyTorch", torch.__version__)
print("Torchvision", torchvision.__version__)
print("Torchattacks", torchattacks.__version__)
print("Numpy", np.__version__)
print("Medmnist", medmnist.__version__)

##Dataset

data_flag =  
[tissuemnist, pathmnist, chestmnist, dermamnist, octmnist, pnemoniamnist, retinamnist, breastmnist, bloodmnist, tissuemnist, organamnist, organcmnist, organsmnist]

In [ ]:
data_flag = 'retinamnist'
# [tissuemnist, pathmnist, chestmnist, dermamnist, octmnist,
# pnemoniamnist, retinamnist, breastmnist, bloodmnist, tissuemnist, organamnist, organcmnist, organsmnist]
download = True

NUM_EPOCHS = 10
BATCH_SIZE = 10
lr = 0.005

info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])

print("number of channels : ", n_channels)
print("number of classes : ", n_classes)

In [ ]:
from torchvision.transforms.transforms import Resize
# preprocessing
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Lambda(lambda image: image.convert('RGB')),
    torchvision.transforms.AugMix(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])
test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Lambda(lambda image: image.convert('RGB')),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# load the data
train_dataset = DataClass(split='train', transform=train_transform, download=download)
test_dataset = DataClass(split='test', transform=test_transform, download=download)

# pil_dataset = DataClass(split='train', download=download)

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader_at_eval = data.DataLoader(dataset=train_dataset, batch_size=2*BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

In [ ]:
print(train_dataset)
print("===================")
print(test_dataset)

##Model

MedViTs ---> QMedViT_Softmax_Only

In [ ]:
from quantum_variants.softmax_only import QMedViT_Softmax_Only

# Training uses the fast analytic simulator path (qpu_mode=False).
# We'll swap in a QPU-style model with shots after training.
model = QMedViT_Softmax_Only(
    stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
    num_classes=n_classes,
    qpu_mode=False,
).cuda()

## Train

In [ ]:
# define loss function and optimizer
if task == "multi-label, binary-class":
    criterion = nn.BCEWithLogitsLoss()
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

In [ ]:
# train

for epoch in range(NUM_EPOCHS):
    train_correct = 0
    train_total = 0
    test_correct = 0
    test_total = 0
    print('Epoch [%d/%d]'% (epoch+1, NUM_EPOCHS))
    model.train()
    for inputs, targets in tqdm(train_loader):
        inputs, targets = inputs.cuda(), targets.cuda()
        # forward + backward + optimize
        optimizer.zero_grad()
        outputs = model(inputs)

        if task == 'multi-label, binary-class':
            targets = targets.to(torch.float32)
            loss = criterion(outputs, targets)
        else:
            targets = targets.squeeze().long()
            loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

## Build QPU-style model for evaluation

In [ ]:
# qpu_mode=True swaps the U-extraction shortcut for the per-sample
# circuit execution that real hardware requires. Here the device is
# still local (default.qubit) but with finite `qpu_shots`, so we get
# simulated shot-noise — a realistic preview of what running on a
# real QPU would give without paying for hardware time.
model_qpu = QMedViT_Softmax_Only(
    stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
    num_classes=n_classes,
    qpu_mode=True,
    qpu_shots=5000,
    qdevice='default.qubit',
).cuda()

# Copy the parameters trained on the analytic simulator. The forward
# is now stochastic (shot noise) so the same image may give slightly
# different logits across runs.
model_qpu.load_state_dict(model.state_dict())
model_qpu.eval()
print('QPU-sim model ready (shots=5000, device=default.qubit)')

##Test

In [ ]:
# QPU-style eval is much slower (per-sample circuit calls instead
# of one matmul over the whole batch). We evaluate on a subset of
# the test set to keep this tractable; bump SUBSET to len(test_dataset)
# if you want full metrics and have ~1-2 hours to spare.
from torch.utils.data import Subset
import time

SUBSET = 50
subset_idx = list(range(min(SUBSET, len(test_dataset))))
subset_loader = data.DataLoader(
    Subset(test_dataset, subset_idx),
    batch_size=2, shuffle=False,
)

split = 'test'
y_score = torch.tensor([])
t0 = time.time()
with torch.no_grad():
    for inputs, targets in tqdm(subset_loader):
        inputs = inputs.cuda()
        outputs = model_qpu(inputs)
        outputs = outputs.softmax(dim=-1)
        y_score = torch.cat((y_score, outputs.cpu()), 0)
elapsed = time.time() - t0
print(f'QPU-sim eval: {len(subset_idx)} samples in {elapsed:.1f}s')

# Note: torchattacks / FGSM / PGD below are NOT compatible with
# qpu_mode (they require differentiable forwards, and the per-sample
# QPU loop with shots is too slow for adversarial iterations anyway).
# Use the analytic `model` for those if you need them.